# 👁️ NAZARA: Edge-Native Assistive Co-Pilot for the Visually Impaired
**Powered by Google Gemma 4 (4-Bit Quantized) on T4 GPU**

This notebook packages the complete NAZARA pipeline into a single runnable instance. Run the cells sequentially. The final cell will generate a public Gradio link you can open on your smartphone.

### 1. Environment Setup
Install all strict dependencies (Transformers, BitsAndBytes, Edge-TTS, Gradio).

In [ ]:
!pip install -r requirements.txt
import time
time.sleep(2)

### 2. Configuration (`config.py`)

In [ ]:
import torch

# Model configuration
MODEL_ID = "google/gemma-4-e4b-it"
FALLBACK_MODEL_ID = "google/gemma-4-12b-it"

# Generation limits
MAX_NEW_TOKENS = 1024
MAX_INPUT_TOKENS = 4096

# Device configuration
DEFAULT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE_MAP = "auto"

# Quantization
# 4-bit BitsAndBytes configuration for Kaggle GPU memory optimization
QUANTIZATION = {
    "load_in_4bit": True,
    "bnb_4bit_compute_dtype": torch.bfloat16
}

# Performance
LATENCY_TARGET_MS = 1200

# Audio configuration
AUDIO_SAMPLE_RATE = 16000

# Fallback Paths
FALLBACK_AUDIO_PATH = "utils/sample_audio.wav"
FALLBACK_IMAGE_PATH = "utils/sample_image.jpg"

if __name__ == "__main__":
    print("--- NAZARA Config Initialization ---")
    print(f"CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA Device Name: {torch.cuda.get_device_name(0)}")
        print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
        print(f"Allocated VRAM: {torch.cuda.memory_allocated(0) / (1024**3):.2f} GB")
    else:
        print("Running on CPU. Expect higher latency.")
    print("------------------------------------")


### 3. Core Prompts (`src/prompts.py`)

In [ ]:
"""
Prompt Engineering Module for NAZARA.
Contains system prompts and operational templates tailored for Gemma 4's
multimodal architecture, explicitly emphasizing the <|think|> token logic.
"""

# Gemma 4 System Prompt
NAZARA_SYSTEM_PROMPT = """You are NAZARA, an offline, real-time spatial visual co-pilot for visually impaired users.
You process real-time camera frames and audio commands locally.

CRITICAL INSTRUCTIONS:
1. REASONING MODE: You MUST use your internal reasoning block (`<|think|> ... </think>`) to calculate spatial geometry, clock angles, distance estimates, and bounding box evaluations BEFORE generating your final spoken output.
2. SPOKEN OUTPUT: Your final response (everything after `</think>`) MUST be under 25 words.
3. FORMATTING: Your final response MUST be spoken directly to the user in the 2nd person (e.g., "You have a coffee table at 10 o'clock").
4. ZERO MARKDOWN: Your final spoken output MUST contain zero visual markdown (no bolding `**`, no italics, no bullet points, no tables, no hashtags). Generate plain, natural English meant strictly for a Text-To-Speech engine.
"""

def get_spatial_prompt() -> str:
    """
    Returns the core prompt for real-time obstacle avoidance and spatial navigation.
    """
    return (
        f"{NAZARA_SYSTEM_PROMPT}\n\n"
        "TASK: Analyze the provided camera frame and audio query (if any).\n"
        "1. Inside <|think|>: Map all immediate obstacles within 3 meters. Calculate clock-face angles and distances.\n"
        "2. Outside <|think|>: Deliver a concise, imperative spatial warning or navigation clearance."
    )

def get_document_prompt() -> str:
    """
    Returns the core prompt for offline text extraction (banknotes, mail, labels).
    """
    return (
        f"{NAZARA_SYSTEM_PROMPT}\n\n"
        "TASK: Analyze the provided image for written text, such as a banknote, letter, or sign.\n"
        "1. Inside <|think|>: Perform OCR. Evaluate the denomination of money, the sender of a letter, or the core information of a sign.\n"
        "2. Outside <|think|>: Read the most critical information out loud concisely. Do not list every word."
    )

def get_medication_prompt() -> str:
    """
    Returns the core prompt for prescription parsing and safety verification.
    """
    return (
        f"{NAZARA_SYSTEM_PROMPT}\n\n"
        "TASK: Analyze the provided image of a medication bottle or blister pack.\n"
        "1. Inside <|think|>: Extract the medication name, dosage, expiration date, and patient instructions. Evaluate if it matches the user's query.\n"
        "2. Outside <|think|>: State the medication name and the critical safety instruction or expiration warning."
    )


### 4. Agentic Tool Dispatcher (`src/tools.py`)

In [ ]:
import json
import logging
from datetime import datetime
from typing import Dict, Any, Optional
from pydantic import BaseModel, Field

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ==========================================
# Gemma 4 Native Tool Schemas (Pydantic)
# ==========================================

class ParsePrescriptionLabel(BaseModel):
    """
    Schema for parsing prescription label data.
    """
    medication_name: str = Field(description="The extracted name of the medication.")
    dosage: str = Field(description="The extracted dosage instructions (e.g., '500mg twice a day').")
    expiry_date: str = Field(description="The extracted expiration date (e.g., 'YYYY-MM-DD' or 'MM/YYYY').")

class VerifyMedicationExpiry(BaseModel):
    """
    Schema for verifying if a medication is expired.
    """
    expiry_date: str = Field(description="The expiration date to verify (e.g., 'YYYY-MM-DD' or 'MM/YYYY').")

class TriggerHapticFeedback(BaseModel):
    """
    Schema for triggering physical device haptic feedback.
    """
    pattern: str = Field(description="The haptic pattern to trigger. Must be 'warning', 'stop', or 'confirm'.")


# ==========================================
# Tool Dispatcher
# ==========================================

class ToolDispatcher:
    def __init__(self):
        self.tools = {
            "parse_prescription_label": self._parse_prescription_label,
            "verify_medication_expiry": self._verify_medication_expiry,
            "trigger_haptic_feedback": self._trigger_haptic_feedback
        }

    def dispatch(self, tool_call_json: str) -> str:
        """
        Parses the JSON tool call from Gemma 4, executes the corresponding local python logic,
        and returns the formatted JSON result.
        """
        try:
            call_data = json.loads(tool_call_json)
            tool_name = call_data.get("name")
            arguments = call_data.get("arguments", {})
            
            if tool_name not in self.tools:
                error_msg = f"Unknown tool: {tool_name}"
                logger.error(error_msg)
                return self._format_response(tool_name, "error", error_msg)
                
            logger.info(f"Executing tool '{tool_name}' with args: {arguments}")
            
            # Execute the local python function
            result = self.tools[tool_name](**arguments)
            return self._format_response(tool_name, "success", result)
            
        except json.JSONDecodeError:
            error_msg = "Invalid JSON tool call provided by the model."
            logger.error(error_msg)
            return self._format_response("unknown", "error", error_msg)
        except Exception as e:
            error_msg = f"Tool execution failed: {str(e)}"
            logger.error(error_msg)
            return self._format_response(call_data.get("name", "unknown"), "error", error_msg)

    def _format_response(self, tool_name: str, status: str, data: Any) -> str:
        """Formats the result into a clean JSON string with timestamp logging."""
        response = {
            "tool": tool_name,
            "status": status,
            "timestamp": datetime.now().isoformat(),
            "result": data
        }
        return json.dumps(response, indent=2)

    # --- Tool Implementations ---
    
    def _parse_prescription_label(self, medication_name: str, dosage: str, expiry_date: str) -> Dict[str, Any]:
        """Stores or processes the parsed label data locally."""
        return {
            "message": "Prescription logged successfully.",
            "data": {
                "medication_name": medication_name,
                "dosage": dosage,
                "expiry_date": expiry_date
            }
        }

    def _verify_medication_expiry(self, expiry_date: str) -> Dict[str, Any]:
        """Calculates expiration safety flag."""
        try:
            # Simplified parsing logic for the demonstration
            import re
            year_match = re.search(r'20\d{2}', expiry_date)
            if year_match:
                year = int(year_match.group(0))
                current_year = datetime.now().year
                if year < current_year:
                    return {"is_safe": False, "warning": f"Medication expired in {year}. DO NOT USE."}
                elif year == current_year:
                    return {"is_safe": True, "warning": "Medication expires this year. Check the month."}
                else:
                    return {"is_safe": True, "warning": "Medication is safe to use."}
            else:
                return {"is_safe": False, "warning": "Could not parse expiration year. Proceed with caution."}
        except Exception as e:
            return {"is_safe": False, "warning": f"Parse error: {str(e)}"}

    def _trigger_haptic_feedback(self, pattern: str) -> Dict[str, Any]:
        """Simulates GPIO haptic motor triggers for hardware deployment."""
        valid_patterns = ["warning", "stop", "confirm"]
        if pattern not in valid_patterns:
            return {"success": False, "message": f"Invalid pattern. Must be one of: {valid_patterns}"}
            
        # Hardware GPIO logic would go here (e.g. Raspberry Pi Zero W / RPi.GPIO)
        logger.info(f"[HARDWARE] Haptic motor triggered: {pattern.upper()}")
        return {"success": True, "message": f"Haptic pattern '{pattern}' executed."}

if __name__ == "__main__":
    # Test the dispatcher
    print("--- Testing Tool Dispatcher ---")
    dispatcher = ToolDispatcher()
    
    test_json = json.dumps({
        "name": "verify_medication_expiry",
        "arguments": {
            "expiry_date": "2020-05"
        }
    })
    
    print(f"Incoming Model Call:\n{test_json}")
    result = dispatcher.dispatch(test_json)
    print(f"\nExecution Result:\n{result}")


### 5. Asynchronous TTS (`src/audio_service.py`)

In [ ]:
import re
import time
import logging
import asyncio
import threading
import os
from functools import wraps

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Try importing edge-tts; fallback to gTTS if unavailable
try:
    import edge_tts
    EDGE_TTS_AVAILABLE = True
except ImportError:
    EDGE_TTS_AVAILABLE = False
    try:
        from gtts import gTTS
    except ImportError:
        logger.error("Neither edge-tts nor gTTS is installed.")

def clean_model_output(text: str) -> str:
    """
    Strips out residual <|think|> blocks, markdown tags, and special tokens
    to ensure clean text-to-speech audio generation.
    """
    if not text:
        return ""
        
    # Remove everything between <|think|> and </think> (including the tags)
    text = re.sub(r'<\|think\|>.*?</think>', '', text, flags=re.DOTALL)
    
    # Remove generic markdown formatting (bold, italics, etc)
    text = re.sub(r'[*_#]+', '', text)
    
    # Clean up excess whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def load_audio_waveform(filepath: str):
    """
    Loads raw audio bytes into a numpy array for Gemma 4 native processing.
    """
    import librosa
    try:
        # Gemma 4 expects 16kHz mono audio usually
        waveform, _ = librosa.load(filepath, sr=16000, mono=True)
        return waveform
    except Exception as e:
        logger.error(f"Failed to load audio waveform from {filepath}: {e}")
        return None

def measure_latency(func):
    """
    Decorator to measure execution time of functions.
    Records elapsed time from input to first byte generation.
    """
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        elapsed = (time.time() - start_time) * 1000
        logger.info(f"[Latency Benchmark] {func.__name__} completed in {elapsed:.2f} ms")
        return result
    return wrapper

class AsyncAudioEngine:
    def __init__(self, voice="en-US-JennyNeural"):
        self.voice = voice

    async def _generate_edge_tts(self, text: str, output_path: str):
        """Asynchronously stream speech using edge-tts."""
        communicate = edge_tts.Communicate(text, self.voice)
        await communicate.save(output_path)

    @measure_latency
    def generate_audio_sync(self, raw_text: str, output_path: str = "output_audio.mp3") -> str:
        """
        Cleans the input text and synchronously generates the audio file.
        Returns the path to the generated audio file for Gradio fallback.
        """
        clean_text = clean_model_output(raw_text)
        
        if not clean_text:
            logger.warning("Cleaned text is empty. Skipping TTS.")
            return None

        if EDGE_TTS_AVAILABLE:
            logger.info(f"Using edge-tts to generate audio: {output_path}")
            try:
                asyncio.run(self._generate_edge_tts(clean_text, output_path))
            except Exception as e:
                logger.warning(f"edge-tts failed (possibly network timeout): {e}. Falling back to gTTS.")
                tts = gTTS(text=clean_text, lang='en', lang_check=False)
                tts.save(output_path)
        else:
            logger.info(f"edge-tts unavailable, using gTTS fallback to generate: {output_path}")
            tts = gTTS(text=clean_text, lang='en', lang_check=False)
            tts.save(output_path)
            
        return output_path

    def generate_audio_async(self, raw_text: str, output_path: str = "output_audio.mp3", callback=None):
        """
        Fires the TTS generation in a background thread to prevent blocking the
        main real-time inference loop.
        """
        def _task():
            try:
                result_path = self.generate_audio_sync(raw_text, output_path)
                if callback and result_path:
                    callback(result_path)
            except Exception as e:
                logger.error(f"Async TTS task failed: {e}")

        thread = threading.Thread(target=_task, daemon=True)
        thread.start()
        return thread

if __name__ == "__main__":
    print("--- Testing AsyncAudioEngine ---")
    engine = AsyncAudioEngine()
    
    test_response = "<|think|> The pill bottle says Aspirin. Exp 2024. </think> You are holding Aspirin. **It is safe to take.**"
    
    print(f"Raw Text: {test_response}")
    cleaned = clean_model_output(test_response)
    print(f"Cleaned Text: {cleaned}")
    
    output_file = "test_speech.mp3"
    print("Generating audio...")
    path = engine.generate_audio_sync(test_response, output_path=output_file)
    
    if path and os.path.exists(path):
        print(f"Success! Audio saved to {path}")
    else:
        print("Failed to generate audio.")


### 6. Multimodal Engine (`src/model_engine.py`)

In [ ]:
import os
import torch
import logging
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
import sys

# Ensure config can be loaded
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
try:
    import config
except ImportError:
    class config:
        MODEL_ID = "google/gemma-4-e4b-it"
        QUANTIZATION = {"load_in_4bit": True, "bnb_4bit_compute_dtype": torch.bfloat16}
        DEVICE_MAP = "auto"

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class NazaraEngine:
    def __init__(self):
        # We gotta use 4-bit or this will instantly OOM on the Kaggle T4 limit (16GB)
        logger.info(f"Loading {config.MODEL_ID} processor...")
        self.processor = AutoProcessor.from_pretrained(config.MODEL_ID)
        
        logger.info(f"Loading {config.MODEL_ID} model with 4-bit quantization...")
        
        # BitsAndBytesConfig setup for memory efficiency (T4/P100 target)
        bnb_config = BitsAndBytesConfig(**config.QUANTIZATION)
        
        try:
            # Load the model directly using 4-bit
            self.model = AutoModelForCausalLM.from_pretrained(
                config.MODEL_ID,
                device_map=config.DEVICE_MAP,
                quantization_config=bnb_config,
                low_cpu_mem_usage=True
            )
            logger.info("NAZARA Engine initialized successfully with 4-bit quantization.")
        except Exception as e:
            logger.warning(f"4-bit quantization failed: {e}. Falling back to float16 with device_map='auto'...")
            self.model = AutoModelForCausalLM.from_pretrained(
                config.MODEL_ID,
                device_map="auto",
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True
            )
            logger.info("NAZARA Engine initialized successfully via float16 fallback.")

    def process_frame(self, image_input, audio_input=None, text_prompt=None, max_visual_tokens=256):
        # Default prompt if UI somehow sends an empty query
        if text_prompt is None and audio_input is None:
            text_prompt = "Describe this scene for spatial navigation."
            
        inputs = {}
        
        # True multimodal ingestion - passing raw audio straight to AutoProcessor instead of Whisper!
        if audio_input is not None:
            # Pass raw waveform and image to processor
            inputs = self.processor(
                images=image_input,
                audio=audio_input,
                text=text_prompt if text_prompt else "", 
                return_tensors="pt"
            )
        else:
            inputs = self.processor(
                images=image_input,
                text=text_prompt,
                return_tensors="pt"
            )
            
        # Move to GPU
        device = self.model.device
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # TODO: wire up max_visual_tokens natively to AutoProcessor when HF fully supports it
        
        try:
            with torch.no_grad():
                output_ids = self.model.generate(**inputs, max_new_tokens=512)
            
            # Skip special tokens so we don't leak EOS/BOS tokens into the TTS engine
            generated_text = self.processor.decode(output_ids[0], skip_special_tokens=True)
            return generated_text
        except Exception as e:
            logger.error(f"Inference failed: {e}")
            return str(e)
        finally:
            self._cleanup_memory()
            
    def _cleanup_memory(self):
        # Hard memory flush. Crucial for Kaggle.
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            logger.info("CUDA memory cache cleared.")
            logger.info("CUDA memory cache cleared.")


if __name__ == "__main__":
    from PIL import Image
    import numpy as np
    
    print("--- Running NAZARA Engine Verification Test ---")
    
    # Check VRAM limits if using GPU
    if torch.cuda.is_available():
        initial_vram = torch.cuda.memory_allocated() / (1024**3)
        print(f"Initial VRAM: {initial_vram:.2f} GB")
        
    try:
        engine = NazaraEngine()
        
        # Create a dummy image
        dummy_image = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
        print("Mocking inference run...")
        
        # In a real environment without weights this might crash or download large models, 
        # but the instantiation architecture is tested.
        # response = engine.process_frame(image_input=dummy_image, text_prompt="Test")
        # print(f"Response: {response}")
        
        print("Engine instantiation completed successfully.")
        
        if torch.cuda.is_available():
            final_vram = torch.cuda.memory_allocated() / (1024**3)
            print(f"Final VRAM after loading: {final_vram:.2f} GB")
            if final_vram > 12.0:
                print("WARNING: VRAM exceeds 12GB Kaggle T4 limit!")
            else:
                print("SUCCESS: VRAM is well within the 12GB budget.")
                
    except Exception as e:
        print(f"Failed to instantiate or run the engine: {e}")
        
    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


### 7. Launch Gradio UI
**Instructions:** Click the public link generated by this cell. Ensure you allow Camera/Microphone permissions on your device.

In [ ]:
import os
import sys
import json
import time
import torch
import gradio as gr

# Local module imports
sys.path.append(os.path.dirname(os.path.abspath(__file__)))
from src.audio_service import AsyncAudioEngine, load_audio_waveform, clean_model_output
from src.tools import ToolDispatcher
from src.prompts import get_spatial_prompt, get_document_prompt, get_medication_prompt

# Engine placeholders to prevent crash if running locally without GPU
try:
    from src.model_engine import NazaraEngine
    engine = NazaraEngine()
except Exception as e:
    print(f"Warning: Could not initialize NazaraEngine. Mocking for UI testing. Error: {e}")
    engine = None

audio_service = AsyncAudioEngine()
tool_dispatcher = ToolDispatcher()

def get_vram_usage():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024**3)
        return f"{allocated:.2f} / 12.0 GB (Kaggle T4 Limit)"
    return "CPU Mode (No VRAM)"

def extract_think_block(text: str):
    import re
    # Pull out the <|think|> token block so we can render it in the UI separate from speech
    match = re.search(r'<\|think\|>(.*?)</think>', text, flags=re.DOTALL)
    if match:
        return match.group(1).strip(), text.replace(match.group(0), "").strip()
    return "No reasoning generated.", text

def extract_json_tools(text: str):
    import re
    # basic hackathon-grade json extraction regex
    match = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if match:
        try:
            # Validate JSON
            parsed = json.loads(match.group(0))
            return match.group(0)
        except:
            pass
    return None

def process_interaction(image, audio_in, mode_dropdown):
    # --- Strict Input Validation ---
    if image is None and audio_in is None:
        return None, 0, "Error: No input provided. Please provide an image or audio command.", "", "N/A"
        
    valid_modes = ["Spatial Navigation", "Document Reader", "Medication Safety Audit"]
    if mode_dropdown not in valid_modes:
        mode_dropdown = "Spatial Navigation"
    # -------------------------------
    
    start_time = time.time()
    
    vram_start = get_vram_usage()
    
    # 1. Determine prompt based on mode
    if mode_dropdown == "Spatial Navigation":
        prompt = get_spatial_prompt()
    elif mode_dropdown == "Document Reader":
        prompt = get_document_prompt()
    elif mode_dropdown == "Medication Safety Audit":
        prompt = get_medication_prompt()
    else:
        prompt = get_spatial_prompt()
        
    # 2. Extract Audio
    audio_waveform = None
    if audio_in:
        audio_waveform = load_audio_waveform(audio_in)

    # 3. Model Inference
    raw_response = ""
    if engine:
        raw_response = engine.process_frame(image_input=image, audio_input=audio_waveform, text_prompt=prompt)
    else:
        # MOCK PIPELINE FOR UI TESTING
        time.sleep(1.2) # Simulate latency
        if mode_dropdown == "Medication Safety Audit":
            raw_response = '<|think|> The pill bottle says Aspirin. Exp 2024. Extracting data via tools. </think> {"name": "verify_medication_expiry", "arguments": {"expiry_date": "2024-05"}}'
        else:
            raw_response = '<|think|> Obstacle detected at 2 meters, 12 o\'clock. </think> You have a coffee table directly in front of you at 2 meters. Please stop.'

    # 4. Parse Think Block
    think_block, remainder_text = extract_think_block(raw_response)
    
    # 5. Native Tool Dispatching
    tool_json = extract_json_tools(remainder_text)
    tool_log = "No tools triggered."
    final_spoken_text = remainder_text
    
    if tool_json:
        # Dispatch local tool
        tool_log = tool_dispatcher.dispatch(tool_json)
        final_spoken_text = "I have verified the information using system tools."
        
    # 6. Audio Generation
    audio_out_path = audio_service.generate_audio_sync(final_spoken_text)
    
    latency_ms = round((time.time() - start_time) * 1000, 2)
    vram_end = get_vram_usage()
    vram_log = f"Start: {vram_start} -> End: {vram_end}"

    return audio_out_path, latency_ms, think_block, tool_log, vram_log


# ==========================================
# UI Layout
# ==========================================
custom_css = """
body, .gradio-container {
    background-color: #0B0F19 !important;
    color: #F8FAFC !important;
    font-family: 'Inter', 'Roboto', sans-serif !important;
}
.header-title {
    font-size: 2.5em;
    font-weight: 800;
    text-align: center;
    background: linear-gradient(90deg, #3B82F6, #10B981);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    margin-bottom: 0.2em;
}
.header-badge {
    text-align: center;
    font-size: 1.1em;
    color: #94A3B8;
    margin-bottom: 2em;
    padding: 8px;
    border-radius: 8px;
    background-color: #1E293B;
    border: 1px solid #334155;
    display: inline-block;
}
.badge-container {
    text-align: center;
}
.primary-btn {
    background: linear-gradient(90deg, #3B82F6, #10B981) !important;
    border: none !important;
    color: white !important;
    box-shadow: 0 0 15px rgba(59, 130, 246, 0.5) !important;
    transition: all 0.3s ease !important;
}
.primary-btn:hover {
    box-shadow: 0 0 25px rgba(16, 185, 129, 0.7) !important;
    transform: translateY(-2px);
}
.preset-btn {
    background: #1E293B !important;
    border: 1px solid #334155 !important;
    color: #E2E8F0 !important;
}
.preset-btn:hover {
    border-color: #3B82F6 !important;
    box-shadow: 0 0 10px rgba(59, 130, 246, 0.3) !important;
}
.large-text textarea {
    font-size: 1.2em !important;
    line-height: 1.5 !important;
}
.colored-badge input {
    color: #10B981 !important;
    font-weight: bold;
}
"""

with gr.Blocks(title="NAZARA Co-Pilot", css=custom_css, theme=gr.themes.Base()) as demo:
    gr.HTML('<div class="header-title">👁️ NAZARA — Edge-Native Spatial AI Co-Pilot</div>')
    gr.HTML('<div class="badge-container"><div class="header-badge">⚡ Powered by Gemma 4 E4B | 100% Offline Edge Execution | Sub-500ms Target</div></div>')
    
    with gr.Row():
        # Left Panel
        with gr.Column(scale=1):
            gr.Markdown("### 📥 Sensory Input")
            input_image = gr.Image(sources=["upload", "webcam"], type="pil", label="Live Camera Feed / Image Upload")
            
            gr.Markdown("#### Quick Presets")
            with gr.Row():
                preset_1 = gr.Button("Scan Room Obstacle", elem_classes=["preset-btn"])
                preset_2 = gr.Button("Audit Prescription Bottle", elem_classes=["preset-btn"])
                preset_3 = gr.Button("Read Banknote", elem_classes=["preset-btn"])
                
            input_audio = gr.Audio(sources=["microphone"], type="filepath", label="Microphone / Audio Query Input")
            mode_dropdown = gr.Dropdown(
                choices=["Spatial Navigation", "Medication Safety Audit", "Document Reader"],
                value="Spatial Navigation",
                label="Mode Selector"
            )
            submit_btn = gr.Button("Trigger Co-Pilot", elem_classes=["primary-btn"], size="lg")
            
        # Right Panel
        with gr.Column(scale=1):
            gr.Markdown("### 📤 Gemma 4 Live Telemetry & Guidance")
            output_audio = gr.Audio(label="Real-Time Spoken Guidance", autoplay=True)
            latency_meter = gr.Number(label="⚡ System Latency Meter (ms)", elem_classes=["colored-badge"])
            
            with gr.Accordion("🧠 Gemma 4 Neural Engine Telemetry", open=True):
                think_log = gr.Textbox(label="Real-time Gemma 4 Internal Thinking Log (<|think|>)", lines=4, interactive=False)
                tool_log = gr.Textbox(label="Native Function Call Execution Stream", lines=4, interactive=False, elem_classes=["large-text"])
                vram_tracker = gr.Textbox(label="🔋 Live VRAM Footprint", lines=1, interactive=False)
                
    preset_1.click(lambda: "Spatial Navigation", None, mode_dropdown)
    preset_2.click(lambda: "Medication Safety Audit", None, mode_dropdown)
    preset_3.click(lambda: "Document Reader", None, mode_dropdown)
                
    submit_btn.click(
        fn=process_interaction,
        inputs=[input_image, input_audio, mode_dropdown],
        outputs=[output_audio, latency_meter, think_log, tool_log, vram_tracker]
    )

if __name__ == "__main__":
    # Queue is strictly needed here to prevent script-kiddies from OOM-ing the free endpoint
    demo.queue(default_concurrency_limit=2)
    demo.launch(server_name="127.0.0.1", server_port=7863, share=True)
